# Lecture 18: ABC-SMC demonstration

Two interactive demonstrations of ABC-SMC as in Lecture 17. Generation 0 is rejection ABC from the prior. Generation $t$ proposes from the weighted mixture

$$q_t(\theta) = \sum_{j} w_{t-1,j}\, N(\theta;\ \theta_{t-1,j},\ \tau^2 I),$$

simulates, keeps $\theta$ if $d(X, x^{obs}) < \epsilon_t$, and gives each accepted particle the weight $\tilde w_{t,i} = \pi(\theta_{t,i}) / q_t(\theta_{t,i})$.

The sampler is in `abcsmc.py`. Some plots have the option to show exact ABC posteriors, but the sampler doesn't use these and they would not usually be available.

Each example starts with **prior draws**: θ from the prior and X from the simulator, with nothing rejected. This shows what rejection ABC has to work with. Then we show ABC-SMC.

**Controls.** Change the settings (kernel τ = how far away a new proposal is from its parent particle, ε = the tolerance threshold for this round of ABC, seed so you can reproduce runs exactly) and press **Reset**. You can change **x_obs** to a different value and it will restart from generation 0. You can generate proposals and go on to the next generation once you think you have enough.

* **+1 proposal** draws one proposal and shows where it came from. **+ batch of** draws as many proposals (not acceptances) as you type. A generation has however many particles you have accepted.
* **Next generation ▸** is available once at least one particle has been accepted. It ends the current generation, weights its particles, and builds $q_{t+1}$ from them, using the tolerance in **with ε =**. It draws nothing from $q_{t+1}$, so you can look at the new proposal first. The box is pre-filled with half the current ε. Type any value to override it.
* **show exact ABC posterior** overlays the exact ABC posterior at the current ε. It is off by default.

For a given seed, the proposals are the same whether you step one at a time or in batches.

In [2]:
from app import PriorDraws, Demo

## Example 1: one parameter, one observation, three explanations

$$\Theta \sim \mathrm{Unif}(-2.5, 2.5), \qquad X  \sim N(f(\theta), \sigma^2), \qquad f(\theta) = \Theta^3 - 3\Theta \qquad x^{obs} = 0, \qquad d(X, x^{obs}) = |X - x^{obs}|.$$

The observation $x^{obs} = 0$ is explained equally well by $\theta \approx -\sqrt3$, $0$ and $\sqrt3$. (A side note about the posterior: The curve is flatter at $0$ ($|f'(0)| = 3$ versus $6$), so for small $\sigma$ and $\epsilon$ the middle solution has about twice the posterior mass: about $0.25, 0.5, 0.25$.)

**Prior draws.** Each draw is a point $(\theta, X)$ scattered around the curve. Rejection ABC will keep only the points within ε of the orange line at $x^{obs}$.

In [3]:
PriorDraws(1)

**Demonstration widget.**


**Top panel:** the noise-free curve $f(\theta) = \theta^3 - 3 \theta$, the band $x^{obs} \pm \epsilon_t$, and each proposal as $(\theta, X)$: blue if accepted, gray if rejected. The most recent proposal is circled in green, with a line from $f(\theta)$ to the simulated $X$. That line is the simulator's noise.

**Bottom panel:** the parameter axis, showing the proposal density $q_t$ (purple), the prior (dotted), and a weighted histogram of the accepted particles. Below the axis is one row of particles per generation: gray for older generations, purple for the previous generation (which defines $q_t$), and blue for the current one. Dot area is proportional to weight.

**Right:** acceptance rate by generation.

In [4]:
demo1 = Demo(1)

## Example 2: two parameters, four modes

$$\Theta_1, \Theta_2 \overset{iid}{\sim} \mathrm{Unif}(-4, 4), \qquad X_i \sim N(\Theta_i^2, \sigma^2), \qquad x^{obs} = (4, 4), \qquad d = \|X - x^{obs}\|_2.$$

The posterior has modes near $(\pm 2, \pm 2)$, and by symmetry each quadrant has posterior mass $1/4$. The exact ABC posterior uses
$P_\theta(d < \epsilon) = F_{\chi'^2_2(\lambda(\theta))}(\epsilon^2/\sigma^2)$, with $\lambda(\theta) = [(\theta_1^2 - 4)^2 + (\theta_2^2 - 4)^2]/\sigma^2$.

These maps are interactive. Every proposal is colored by the distance $d(X, x^{obs})$ between its simulated data and the observed data, from dark (close) to light (far), on a log scale. Accepted particles have an orange outline and area ∝ weight; rejected proposals have no outline. Hover over any point to see $\theta$, the simulated $X$, the observed $x^{obs}$, the distance and ε, and the decision and weight.

**Particles view:** this generation's proposals and particles. **Proposal view:** $q_t$ (purple shading) and the previous generation's particles that define it (purple circles), with the same proposals on top.
**Right:** acceptance rate by generation.

**Prior draws.** Left: $\theta$ from the prior. Right: the simulated data. Only draws that land near $x^{obs} = (4, 4)$ will survive rejection, and those come from all four quadrants of $\theta$, since the sign of each $\theta_i$ doesn't affect $X$.

In [5]:
PriorDraws(2)

**ABC-SMC.**

In [6]:
demo2 = Demo(2)

## Things to notice

- If you have a large kernel standard deviatio, you reject more simulations.
- If you make the kernel variance too small, you can make it so that your proposals don't fill in the space between the particles adequately, and it can't be corrected away by the importance sampling weights unless you have a huge number of samples.
- If you don't have enough particles in a generation, you can miss one or more true modes.
- Notice that you can start off with smaller numbers of particles and only take a large number of samples at the very end (as long as you have enough particles to represent all the modes!).
- Notice that if you decrease ε really dramatically in one generation, your acceptance rate will go way down.